In [1]:
from datasets import load_dataset
from accelerate import Accelerator
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

W0410 20:40:11.946000 28528 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
device = Accelerator().device
device

device(type='cuda')

In [3]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
device

device(type='cuda')

In [4]:
system_prompt = """Ты — лингвист-аналитик. Твоя задача: найти общий смысл в сообщениях.
ПРАВИЛА:
1. Ответ строго в формате: [обобщённый смысл]
2. Никаких лишних слов, пояснений и вежливости.
3. Не пересказывай детали.
4. Обобщай людей до понятий (личность, знакомые, коллектив).
5. Будь максимально кратким."""

In [75]:
def preprocess_function(example):
    return {
        'prompt': [
            {'role': 'user', 'content': example['instruction']}
        ],
        'completion': [
            {'role': 'model', 'content': example['response']}
        ]
    }

In [76]:
dataset_path = '../../data/train_data/msg_clusters.csv'
data = load_dataset('csv', data_files=dataset_path, split='train')
dataset = data.map(preprocess_function, remove_columns=['instruction', 'response'])
dataset = dataset.train_test_split(test_size=0.2)

In [7]:
torch_dtype = torch.bfloat16
model_id = 'google/gemma-3-270m-it'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch_dtype, device_map='auto')

In [8]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [9]:
print(f'Num parameters: {model.num_parameters()}')
print(f'Trainable: {model.num_parameters(only_trainable=True)}')

Num parameters: 268098176
Trainable: 268098176


In [3]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

In [11]:
peft_config = LoraConfig(
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

In [12]:
args = SFTConfig(
    output_dir='../../models/adapters/gemma3_270m_peft_ft',
    max_length=512,
    packing=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4, 
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='linear',
    warmup_ratio=0.1,
    optim='adamw_torch_fused',
    logging_strategy='epoch',
    eval_strategy='epoch',
    max_grad_norm=1.0,
    fp16=True if model.dtype == torch.float16 else False,
    bf16=True if model.dtype == torch.bfloat16 else False,
    dataset_kwargs={
        'add_special_tokens': False,
        'append_concat_token': True
    }
)

In [13]:
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=peft_config,
    processing_class=tokenizer
)

Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-fla

Tokenizing train dataset:   0%|          | 0/608 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/608 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/152 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/152 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Denis\_netrc.
wandb: Currently logged in as: salnikovdenis44 (salnikovdenis44-omstu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.972200,0.784429,3.186791,57439.000000,0.816073
2,0.673400,0.645009,2.761624,114878.000000,0.849089
3,0.433500,0.603270,2.550417,172317.000000,0.844538
4,0.326000,0.645102,2.469247,229756.000000,0.840592
5,0.249800,0.632038,2.411672,287195.000000,0.842752


TrainOutput(global_step=145, training_loss=0.9309684358794114, metrics={'train_runtime': 656.9326, 'train_samples_per_second': 0.868, 'train_steps_per_second': 0.221, 'total_flos': 176150212343040.0, 'train_loss': 0.9309684358794114, 'epoch': 5.0})

In [16]:
trainer.save_model()

In [17]:
del model
del trainer
torch.cuda.empty_cache()

# Сохранение merged модели с адаптером и токенизатора

In [4]:
from peft import PeftModel

In [21]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch_dtype, device_map='auto', low_cpu_mem_usage=True)

In [2]:
model_path = '../../models/gemma3_270m_peft_ft'

In [22]:
peft_model = PeftModel.from_pretrained(model, args.output_dir)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(model_path, safe_serialization=True, max_shard_size='2GB')

In [23]:
processor = AutoTokenizer.from_pretrained(args.output_dir)
processor.save_pretrained(model_path)

('../../models/gemma3_270m_peft_ft\\tokenizer_config.json',
 '../../models/gemma3_270m_peft_ft\\special_tokens_map.json',
 '../../models/gemma3_270m_peft_ft\\chat_template.jinja',
 '../../models/gemma3_270m_peft_ft\\tokenizer.model',
 '../../models/gemma3_270m_peft_ft\\added_tokens.json',
 '../../models/gemma3_270m_peft_ft\\tokenizer.json')

# Загрузка модели

In [5]:
from transformers import pipeline

In [8]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map='auto', dtype='auto')
tokenizer = AutoTokenizer.from_pretrained(model_path)

The tokenizer you are loading from '../../models/gemma3_270m_peft_ft' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [9]:
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [10]:
def get_promt_for_llm(user_question):
    messages = [
        {'role': 'user', 'content': user_question}
    ]
    prompt = pipe.tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return prompt

In [64]:
task = """Что объединяет эти сообщения:
1) 👢
2) 🧸
3) 🤯🤯🤯🤯
4) 🎻🎻🎻
5) 🧓
6) 👺👺👺👺👺
7) 👺
8) 🪓
"""

In [65]:
prompt = get_promt_for_llm(task)
output = pipe(prompt)

In [66]:
print(f"Ответ: {output[0]['generated_text'][len(prompt):].strip()}")

Ответ: смайлики


In [80]:
model_unpretrained = AutoModelForCausalLM.from_pretrained('google/gemma-3-270m-it', dtype='auto', device_map='auto')
tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-270m-it')

In [45]:
pipe_non_pretrained = pipeline('text-generation', model=model_unpretrained, tokenizer=tokenizer)

Device set to use cuda:0


In [67]:
prompt = get_promt_for_llm(task)
output = pipe_non_pretrained(prompt)

In [68]:
print(f"Ответ: {output[0]['generated_text'][len(prompt):].strip()}")

Ответ: Правильный ответ: 1) 👢


# Загрузка на hf

In [7]:
from huggingface_hub import HfApi, login, create_repo

In [5]:
login()

In [8]:
repo_id = "slnkvdns/gemma-270m-it-topic-clustering-ru"

In [9]:
create_repo(repo_id, repo_type="model", exist_ok=True)

RepoUrl('https://huggingface.co/slnkvdns/gemma-270m-it-topic-clustering-ru', endpoint='https://huggingface.co', repo_type='model', repo_id='slnkvdns/gemma-270m-it-topic-clustering-ru')

In [10]:
api = HfApi()
api.upload_folder(
    folder_path=model_path,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/slnkvdns/gemma-270m-it-topic-clustering-ru/commit/f9acb5e46065e15df38b97b7a3050930e37ac649', commit_message='Upload folder using huggingface_hub', commit_description='', oid='f9acb5e46065e15df38b97b7a3050930e37ac649', pr_url=None, repo_url=RepoUrl('https://huggingface.co/slnkvdns/gemma-270m-it-topic-clustering-ru', endpoint='https://huggingface.co', repo_type='model', repo_id='slnkvdns/gemma-270m-it-topic-clustering-ru'), pr_revision=None, pr_num=None)